### Ocupação Hospitalar - COVID19

Ocupacao hospitalar COVID-19: arquivos oficiais localizados de 2020 a 2022.
%pip install pandas pyarrow
%run baixar_ocupacao_hospitalar_2020_2022.py
Fonte: https://dadosabertos.saude.gov.br/dataset/registro-de-ocupacao-hospitalar-covid-19
2023/2024 nao localizados nesta colecao: nao representam zero ocupacao.
Preserva todos os registros, campos e valores como texto, inclusive excluido,
validado e CNES original. Coluna inicial sem nome vira indice_origem_csv.
Nao deduplica nem soma ocupacoes. Registros nao sao pacientes unicos.
Campos de ocupacao de diferentes layouts nao devem ser somados automaticamente.
Reutiliza Parquets completos validados. Para nova coleta, altere PASTA.


In [1]:
import csv
import hashlib
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
BASE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
PASTA = BASE / 'dados_ocupacao_hospitalar'
ANOS = (2020, 2021, 2022)
LOTE = 50000
URLS = {a: f'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/LEITOS/2026-08-16/esus-vepi.LeitoOcupacao_{a}.csv' for a in ANOS}

In [3]:
def salvar_json(p, obj):
    tmp = p.with_suffix(p.suffix+'.part')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(p)


def sha256(p):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1024**2), b''): h.update(b)
    return h.hexdigest()


def baixar(url, destino):
    parcial = destino.with_suffix('.csv.part')
    for tentativa in range(1,4):
        try:
            req = Request(url, headers={'User-Agent':'Observatorio-COVID-Brasil/1.0','Accept-Encoding':'identity'})
            with urlopen(req, timeout=120) as r, parcial.open('wb') as f:
                if r.status != 200: raise IOError(f'HTTP {r.status}: esperado arquivo completo.')
                if 'html' in r.headers.get('Content-Type','').lower(): raise ValueError('Resposta HTML inesperada.')
                total = int(r.headers.get('Content-Length',0)); n = 0; ultimo = time.monotonic()
                while True:
                    b = r.read(1024**2)
                    if not b: break
                    f.write(b); n += len(b)
                    if time.monotonic()-ultimo >= 3:
                        print(f'  {n/1024**2:.1f} MB baixados', flush=True); ultimo = time.monotonic()
                meta = {'etag':r.headers.get('ETag'),'last_modified':r.headers.get('Last-Modified')}
            if not n or (total and n != total): raise IOError('Download vazio/incompleto.')
            parcial.replace(destino)
            return meta
        except Exception as e:
            print(f'Tentativa {tentativa}/3: {e}', flush=True)
            if tentativa == 3: raise
            time.sleep(3*tentativa)


def converter(origem, destino, ano):
    # Cabecalhos e UTF-8 conferidos em amostras reais dos tres anos.
    with origem.open(encoding='utf-8-sig', newline='') as f:
        originais = next(csv.reader(f))
    colunas = [('indice_origem_csv' if i==0 and c=='' else c) for i,c in enumerate(originais)]
    if not all(colunas) or len(set(colunas)) != len(colunas): raise ValueError('Cabecalho vazio/duplicado.')
    if not {'_id','dataNotificacao','cnes','excluido','validado'}.issubset(colunas): raise ValueError('Layout inesperado.')
    parcial = destino.with_suffix('.parquet.part')
    esquema = pa.schema([(c,pa.string()) for c in colunas])
    total = invalidas = fora = 0; minimo = maximo = None
    flags = {'excluido':{}, 'validado':{}}
    with pq.ParquetWriter(parcial, esquema, compression='snappy') as w:
        with origem.open(encoding='utf-8-sig', newline='') as f:
            leitor = csv.reader(f, strict=True); next(leitor)
            lote = []
            def gravar(linhas):
                nonlocal total, invalidas, fora, minimo, maximo
                if not linhas: return
                df = pd.DataFrame(linhas, columns=colunas, dtype='string')
                datas = pd.to_datetime(df['dataNotificacao'], format='ISO8601', errors='coerce', utc=True)
                invalidas += int(datas.isna().sum()); fora += int((datas.notna() & datas.dt.year.ne(ano)).sum())
                if datas.notna().any():
                    lo, hi = datas.min(), datas.max()
                    minimo = lo if minimo is None else min(minimo,lo)
                    maximo = hi if maximo is None else max(maximo,hi)
                for campo in flags:
                    for valor,n in df[campo].value_counts().items(): flags[campo][str(valor)] = flags[campo].get(str(valor),0)+int(n)
                w.write_table(pa.Table.from_pandas(df, schema=esquema, preserve_index=False))
                total += len(df); print(f'  {ano}: {total:,} registros convertidos', flush=True)
            for linha in leitor:
                if not linha: continue
                if len(linha)!=len(colunas): raise ValueError(f'CSV linha {leitor.line_num}: quantidade de campos inesperada.')
                lote.append(linha)
                if len(lote)>=LOTE: gravar(lote); lote.clear()
            gravar(lote)
    if not total or pq.read_metadata(parcial).num_rows != total: raise ValueError('Parquet vazio/contagem divergente.')
    parcial.replace(destino)
    return {'registros':total, 'colunas':len(colunas), 'datas_invalidas_ou_ausentes':invalidas,
        'datas_de_outro_ano_utc':fora,'data_minima_utc':str(minimo),'data_maxima_utc':str(maximo),
        'contagens_flags_originais':flags,'mapeamento_colunas':list(zip(originais,colunas))}


def main():
    for sub in ('brutos','parquet'): (PASTA/sub).mkdir(parents=True,exist_ok=True)
    manifesto = PASTA/'manifesto_download.json'
    estados = json.loads(manifesto.read_text(encoding='utf-8')) if manifesto.exists() else {}
    salvar_json(PASTA/'cobertura_fontes.json', {'anos_projeto':list(range(2020,2025)),
        'anos_com_arquivo_localizado':list(ANOS),'anos_nao_localizados':[2023,2024],
        'nota':'Ausencia de arquivo nao significa ocupacao zero; recorte dos arquivos nao garante cobertura diaria completa.',
        'fonte':'https://dadosabertos.saude.gov.br/dataset/registro-de-ocupacao-hospitalar-covid-19'})
    falhas = []
    for ano in ANOS:
        url = URLS[ano]; origem = PASTA/'brutos'/f'ocupacao_{ano}.csv'; destino = PASTA/'parquet'/f'ocupacao_{ano}.parquet'
        e = estados.get(str(ano),{})
        try:
            if e.get('status')=='concluido' and e.get('url')==url and destino.exists():
                if sha256(destino)==e.get('sha256_parquet') and pq.read_metadata(destino).num_rows==e.get('registros'):
                    print(f'{ano}: concluido e validado.',flush=True); continue
            print(f'\n{ano}: baixando arquivo oficial...',flush=True)
            meta = baixar(url,origem); info = converter(origem,destino,ano)
            estados[str(ano)] = {'status':'concluido','url':url,'coletado_utc':datetime.now(timezone.utc).isoformat(),
                'sha256_csv':sha256(origem),'sha256_parquet':sha256(destino),**meta,**info}
        except Exception as e:
            falhas.append(ano); estados[str(ano)] = {'status':'erro','url':url,'erro':str(e)}
            print(f'Falha em {ano}: {e}',flush=True)
        salvar_json(manifesto,estados)
    if falhas: raise RuntimeError(f'Anos com falha: {falhas}. Consulte {manifesto}. Execute novamente para tentar esses anos.')
    print('\nConcluido para os arquivos localizados (2020–2022): '+str(PASTA/'parquet'))
    print('2023/2024 nao localizados. Flags excluido/validado preservadas para revisao antes da analise.')
    return estados


if __name__ == '__main__':
    resultado_ocupacao = main()



2020: baixando arquivo oficial...
  41.0 MB baixados
  87.0 MB baixados
  2020: 50,000 registros convertidos
  2020: 100,000 registros convertidos
  2020: 150,000 registros convertidos
  2020: 200,000 registros convertidos
  2020: 250,000 registros convertidos
  2020: 300,000 registros convertidos
  2020: 350,000 registros convertidos
  2020: 400,000 registros convertidos
  2020: 450,000 registros convertidos
  2020: 500,000 registros convertidos
  2020: 550,000 registros convertidos
  2020: 554,706 registros convertidos

2021: baixando arquivo oficial...
  35.0 MB baixados
  78.0 MB baixados
  122.0 MB baixados
  2021: 50,000 registros convertidos
  2021: 100,000 registros convertidos
  2021: 150,000 registros convertidos
  2021: 200,000 registros convertidos
  2021: 250,000 registros convertidos
  2021: 300,000 registros convertidos
  2021: 350,000 registros convertidos
  2021: 400,000 registros convertidos
  2021: 450,000 registros convertidos
  2021: 500,000 registros convertidos
